# Hierarchical Multiple Instance Learning

This notebook implements a hierarchical approach to the classification
task. Rather than predicting all classes at once, the problem is broken
down into a tree of simpler decisions, each handled by a dedicated node.
Every node is trained independently and the individual predictions are
then combined through a pipeline to produce the final label.

Each node relies on a Multiple Instance Learning formulation, treating a
dialogue as a collection of turns and aggregating them through an
attention mechanism on top of a pre-trained transformer encoder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
from transformers import AutoTokenizer, AutoModel
from datasets import load_from_disk
from tqdm.auto import tqdm

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_PATH = "/content/drive/MyDrive/{hide}"

def set_seed(seed):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  torch.cuda.manual_seed_all(seed)

Pre-computation

In [ ]:
def extract_seeker_turns(dialogue):
    """Estrae solo i testi degli ultimi 3 turni del seeker dal dialogo."""
    turns = [t["text"] for t in dialogue if t["speaker"] == "seeker"]
    return turns[-3:]

def precompute_embeddings(samples, encoder_model, batch_size=16):
    all_turns = []
    sample_offsets = []

    for s in samples:
        turns = extract_seeker_turns(s["dialogue"])
        start = len(all_turns)
        all_turns.extend(turns)
        end = len(all_turns)
        sample_offsets.append((start, end))

    # Encode
    all_embeddings = encoder_model.encode(
        all_turns,
        batch_size=batch_size,
        convert_to_tensor=True,
        device=DEVICE,
    )

    sample_embeddings = []
    for start, end in sample_offsets:
        sample_embeddings.append(all_embeddings[start:end].cpu())

    return sample_embeddings

Dataset

In [ ]:
# Dataset per poter fare MIL

class DialogueBagDataset(Dataset):
    def __init__(self, embeddings, labels):
        assert len(embeddings) == len(labels)
        self.embeddings = embeddings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]


def collate_bags(batch):
    embeddings_list, labels = zip(*batch)
    lengths = torch.tensor([e.shape[0] for e in embeddings_list], dtype=torch.long)

    # Pad: shape (B, T_max, D)
    padded = pad_sequence(embeddings_list, batch_first=True, padding_value=0.0)

    # Mask: shape (B, T_max)
    max_len = padded.shape[1]
    mask = torch.arange(max_len)[None, :] >= lengths[:, None]

    labels = torch.tensor(labels, dtype=torch.long)
    return padded, mask, labels


Model

In [ ]:
# Model with gated attention + linear classifier
class GatedAttentionMIL(nn.Module):
    def __init__(self, embedding_dim, n_classes, hidden_dim=128, dropout=0.1):
        super().__init__()
        self.V = nn.Linear(embedding_dim, hidden_dim)
        self.U = nn.Linear(embedding_dim, hidden_dim)
        self.w = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(embedding_dim, n_classes)

    def forward(self, x, mask):
        v = torch.tanh(self.V(x)) # (B, T, L)
        u = torch.sigmoid(self.U(x)) # (B, T, L)
        gated = v * u # (B, T, L)
        scores = self.w(gated).squeeze(-1) # (B, T)

        scores = scores.masked_fill(mask, float("-inf"))
        attn = F.softmax(scores, dim=1) # (B, T)

        # Pooling
        bag_embedding = (attn.unsqueeze(-1) * x).sum(dim=1) # (B, D)
        bag_embedding = self.dropout(bag_embedding)

        logits = self.classifier(bag_embedding)
        return logits, attn

# "Wrapper" for huggingface models (for convenience)
class HFMeanPoolEncoder:
    def __init__(self, model_name, device, max_length=512):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device)
        self.model.eval()
        self.device = device
        self.max_length = max_length
        self._embedding_dim = self.model.config.hidden_size

    def get_sentence_embedding_dimension(self):
        return self._embedding_dim

    @torch.no_grad()
    def encode(self, texts, batch_size=64,
               convert_to_tensor=True, device=None):

        device = device or self.device
        all_embeddings = []

        iterator = range(0, len(texts), batch_size)

        for i in tqdm(iterator, desc="Encoding: "):
            batch = texts[i:i + batch_size]
            # Empty strings check
            batch = [t if t.strip() else " " for t in batch]

            encoded = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt",
            ).to(device)

            outputs = self.model(**encoded)
            token_embeddings = outputs.last_hidden_state  # (B, T, D)
            attention_mask = encoded["attention_mask"].unsqueeze(-1).float()  # (B, T, 1)

            sum_embeddings = (token_embeddings * attention_mask).sum(dim=1)
            sum_mask = attention_mask.sum(dim=1).clamp(min=1e-9)
            mean_embeddings = sum_embeddings / sum_mask  # (B, D)

            all_embeddings.append(mean_embeddings.cpu())

        embeddings = torch.cat(all_embeddings, dim=0)
        if convert_to_tensor:
            return embeddings
        return embeddings.numpy()


def load_encoder(model_name, device=DEVICE):
    encoder = HFMeanPoolEncoder(model_name, device=device)
    return encoder

Training

In [ ]:
def compute_class_weights(labels, n_classes):
    classes = np.arange(n_classes)
    weights = compute_class_weight("balanced", classes=classes, y=labels)
    return torch.tensor(weights, dtype=torch.float32)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for padded, mask, labels in loader:
        padded, mask = padded.to(device), mask.to(device)
        logits, _ = model(padded, mask)
        preds = logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return macro_f1, np.array(all_preds), np.array(all_labels)

# train a model with early stopping on macro-F1

def train_node(
    train_embeddings, train_labels,
    val_embeddings, val_labels,
    n_classes,
    embedding_dim=768,
    hidden_dim=128,
    dropout=0.1,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=16,
    max_epochs=100,
    patience=10,
    use_class_weights=True,
    seed=42,
    verbose=True,
):
    set_seed(seed)

    train_ds = DialogueBagDataset(train_embeddings, train_labels)
    val_ds = DialogueBagDataset(val_embeddings, val_labels)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_bags)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_bags)

    model = GatedAttentionMIL(embedding_dim, n_classes, hidden_dim, dropout).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    if use_class_weights:
        cw = compute_class_weights(train_labels, n_classes).to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=cw)
        if verbose:
            print(f"Class weights: {cw.cpu().numpy().round(3).tolist()}")
    else:
        criterion = nn.CrossEntropyLoss()

    best_val_f1 = -1
    best_state = None
    epochs_no_improve = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        model.train()
        total_loss = 0.0
        for padded, mask, labels in train_loader:
            padded, mask, labels = padded.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
            logits, _ = model(padded, mask)
            loss = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * labels.size(0)
        train_loss = total_loss / len(train_ds)

        val_f1, _, _ = evaluate(model, val_loader, DEVICE)
        history.append((epoch, train_loss, val_f1))

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
            marker = " ★"
        else:
            epochs_no_improve += 1
            marker = ""

        if verbose:
            print(f"Epoch {epoch:3d} | train_loss={train_loss:.4f} | val_macroF1={val_f1:.4f}{marker}")

        if epochs_no_improve >= patience:
            if verbose:
                print(f"Early stopping at epoch {epoch} (best val F1={best_val_f1:.4f})")
            break

    return best_state, best_val_f1, history


def evaluate_node_full(model_state, val_embeddings, val_labels, n_classes,
                       embedding_dim=768, hidden_dim=128, label_names=None):
    """Carica il best model e valuta su val con report completo."""
    model = GatedAttentionMIL(embedding_dim, n_classes, hidden_dim).to(DEVICE)
    model.load_state_dict(model_state)

    val_ds = DialogueBagDataset(val_embeddings, val_labels)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, collate_fn=collate_bags)

    macro_f1, preds, labels = evaluate(model, val_loader, DEVICE)

    print(f"\n=== Final eval ===")
    print(f"Macro-F1: {macro_f1:.4f}")

    return macro_f1, preds, labels

Encoder

In [ ]:
ENCODER_NAME = "mental/mental-bert-base-uncased"


encoder = load_encoder(ENCODER_NAME)
EMBEDDING_DIM = encoder.get_sentence_embedding_dimension()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: mental/mental-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
pooler.dense.bias                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your d

##N1


In [ ]:
n1_train = load_from_disk(os.path.join(DATA_PATH, "node/noaugmented/n1_3way_train"))
n1_val = load_from_disk(os.path.join(DATA_PATH, "node/noaugmented/n1_3way_val"))

n1_train_emb = precompute_embeddings(n1_train, encoder)
n1_val_emb = precompute_embeddings(n1_val, encoder)

n1_train_labels = [s["label"] for s in n1_train]
n1_val_labels = [s["label"] for s in n1_val]

Encoding:   0%|          | 0/318 [00:00<?, ?it/s]

Encoding:   0%|          | 0/24 [00:00<?, ?it/s]

In [ ]:
print("n1_train labels:", Counter(n1_train_labels))
print("n1_val labels:",   Counter(n1_val_labels))

n1_train labels: Counter({2: 1524, 0: 281, 1: 40})
n1_val labels: Counter({2: 105, 1: 15, 0: 15})


In [ ]:
n1_state, n1_best_f1, n1_history = train_node(
    train_embeddings=n1_train_emb,
    train_labels=n1_train_labels,
    val_embeddings=n1_val_emb,
    val_labels=n1_val_labels,
    n_classes=3,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=128,
    lr=1e-3,
    batch_size=16,
    max_epochs=150,
    patience=18,
    use_class_weights=False,
    seed=0,
)

Epoch   1 | train_loss=0.4322 | val_macroF1=0.4550 ★
Epoch   2 | train_loss=0.2997 | val_macroF1=0.4794 ★
Epoch   3 | train_loss=0.2653 | val_macroF1=0.4489
Epoch   4 | train_loss=0.2397 | val_macroF1=0.4459
Epoch   5 | train_loss=0.2287 | val_macroF1=0.4459
Epoch   6 | train_loss=0.2043 | val_macroF1=0.4241
Epoch   7 | train_loss=0.2018 | val_macroF1=0.4559
Epoch   8 | train_loss=0.1890 | val_macroF1=0.4530
Epoch   9 | train_loss=0.1730 | val_macroF1=0.4530
Epoch  10 | train_loss=0.1680 | val_macroF1=0.4405
Epoch  11 | train_loss=0.1671 | val_macroF1=0.4530
Epoch  12 | train_loss=0.1594 | val_macroF1=0.4479
Epoch  13 | train_loss=0.1508 | val_macroF1=0.4580
Epoch  14 | train_loss=0.1447 | val_macroF1=0.4610
Epoch  15 | train_loss=0.1352 | val_macroF1=0.4442
Epoch  16 | train_loss=0.1312 | val_macroF1=0.4604
Epoch  17 | train_loss=0.1431 | val_macroF1=0.4610
Epoch  18 | train_loss=0.1287 | val_macroF1=0.4587
Epoch  19 | train_loss=0.1239 | val_macroF1=0.4587
Epoch  20 | train_loss=0.12

In [ ]:
n1_f1, n1_preds, n1_true = evaluate_node_full(
    n1_state, n1_val_emb, n1_val_labels,
    n_classes=3,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=128,
    label_names=["0", "1", "2"],
)


=== Final eval ===
Macro-F1: 0.4794


##N2


In [ ]:
n2_train = load_from_disk(os.path.join(DATA_PATH, "node/noaugmented/n2_train"))
n2_val = load_from_disk(os.path.join(DATA_PATH, "node/noaugmented/n2_val"))

n2_train_emb = precompute_embeddings(n2_train, encoder)
n2_val_emb = precompute_embeddings(n2_val, encoder)

n2_train_labels = [s["label"] for s in n2_train]
n2_val_labels = [s["label"] for s in n2_val]

print(f"[N2] Train: {len(n2_train_emb)}, distribution: {Counter(n2_train_labels)}")
print(f"[N2] Val:   {len(n2_val_emb)}, distribution: {Counter(n2_val_labels)}")

Encoding:   0%|          | 0/268 [00:00<?, ?it/s]

Encoding:   0%|          | 0/20 [00:00<?, ?it/s]

[N2] Train: 1524, distribution: Counter({4: 953, 3: 157, 2: 154, 1: 140, 0: 120})
[N2] Val:   105, distribution: Counter({2: 30, 1: 30, 3: 15, 0: 15, 4: 15})


In [ ]:
n2_state, n2_best_f1, n2_history = train_node(
    train_embeddings=n2_train_emb,
    train_labels=n2_train_labels,
    val_embeddings=n2_val_emb,
    val_labels=n2_val_labels,
    n_classes=5,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=128,
    lr=1e-3,
    batch_size=16,
    max_epochs=150,
    patience=20,
    use_class_weights=True,
    seed=987654321,

)

[Train] Class weights: [2.5399999618530273, 2.177000045776367, 1.9789999723434448, 1.940999984741211, 0.3199999928474426]
Epoch   1 | train_loss=1.5650 | val_macroF1=0.1172 ★
Epoch   2 | train_loss=1.4584 | val_macroF1=0.2579 ★
Epoch   3 | train_loss=1.3748 | val_macroF1=0.2614 ★
Epoch   4 | train_loss=1.3154 | val_macroF1=0.2544
Epoch   5 | train_loss=1.2512 | val_macroF1=0.3223 ★
Epoch   6 | train_loss=1.1984 | val_macroF1=0.2579
Epoch   7 | train_loss=1.1690 | val_macroF1=0.3583 ★
Epoch   8 | train_loss=1.1329 | val_macroF1=0.3078
Epoch   9 | train_loss=1.0969 | val_macroF1=0.3368
Epoch  10 | train_loss=1.0674 | val_macroF1=0.3034
Epoch  11 | train_loss=1.0558 | val_macroF1=0.2952
Epoch  12 | train_loss=1.0390 | val_macroF1=0.2602
Epoch  13 | train_loss=1.0001 | val_macroF1=0.3349
Epoch  14 | train_loss=0.9976 | val_macroF1=0.3247
Epoch  15 | train_loss=0.9670 | val_macroF1=0.3194
Epoch  16 | train_loss=0.9464 | val_macroF1=0.3343
Epoch  17 | train_loss=0.9264 | val_macroF1=0.3058
E

In [ ]:
n2_f1, n2_preds, n2_true = evaluate_node_full(
    n2_state, n2_val_emb, n2_val_labels,
    n_classes=5,
    embedding_dim=EMBEDDING_DIM,
    label_names=["1", "ImgDist", "RealAv", "6", "7"],
)


=== Final eval ===
Macro-F1: 0.3583


##N3

In [ ]:
n3_train = load_from_disk(os.path.join(DATA_PATH, "node/n3_train"))
n3_val = load_from_disk(os.path.join(DATA_PATH, "node/n3_val"))

n3_train_emb = precompute_embeddings(n3_train, encoder)
n3_val_emb = precompute_embeddings(n3_val, encoder)

n3_train_labels = [s["label"] for s in n3_train]
n3_val_labels = [s["label"] for s in n3_val]

print(f"[N3] Train: {len(n3_train_emb)}, distribution: {Counter(n3_train_labels)}")
print(f"[N3] Val:   {len(n3_val_emb)}, distribution: {Counter(n3_val_labels)}")

Encoding:   0%|          | 0/26 [00:00<?, ?it/s]

Encoding:   0%|          | 0/6 [00:00<?, ?it/s]

[N3] Train: 140, distribution: Counter({1: 70, 0: 70})
[N3] Val:   30, distribution: Counter({0: 15, 1: 15})


In [ ]:
n3_state, n3_best_f1, n3_history = train_node(
    train_embeddings=n3_train_emb,
    train_labels=n3_train_labels,
    val_embeddings=n3_val_emb,
    val_labels=n3_val_labels,
    n_classes=2,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=128,
    lr=1e-3,
    batch_size=16,
    max_epochs=100,
    patience=10,
    use_class_weights=True,
    seed=0,
)

[Train] Class weights: [1.0, 1.0]
Epoch   1 | train_loss=0.6998 | val_macroF1=0.3651 ★
Epoch   2 | train_loss=0.6623 | val_macroF1=0.5982 ★
Epoch   3 | train_loss=0.6351 | val_macroF1=0.5928
Epoch   4 | train_loss=0.6010 | val_macroF1=0.5928
Epoch   5 | train_loss=0.5693 | val_macroF1=0.5417
Epoch   6 | train_loss=0.5465 | val_macroF1=0.5928
Epoch   7 | train_loss=0.5053 | val_macroF1=0.6652 ★
Epoch   8 | train_loss=0.4770 | val_macroF1=0.5833
Epoch   9 | train_loss=0.4393 | val_macroF1=0.6667 ★
Epoch  10 | train_loss=0.4145 | val_macroF1=0.6667
Epoch  11 | train_loss=0.3876 | val_macroF1=0.6667
Epoch  12 | train_loss=0.3629 | val_macroF1=0.7285 ★
Epoch  13 | train_loss=0.3481 | val_macroF1=0.7285
Epoch  14 | train_loss=0.3241 | val_macroF1=0.6667
Epoch  15 | train_loss=0.3110 | val_macroF1=0.7285
Epoch  16 | train_loss=0.3035 | val_macroF1=0.7285
Epoch  17 | train_loss=0.2946 | val_macroF1=0.6667
Epoch  18 | train_loss=0.2792 | val_macroF1=0.6997
Epoch  19 | train_loss=0.2680 | val_ma

In [ ]:
n3_f1, n3_preds, n3_true = evaluate_node_full(
    n3_state, n3_val_emb, n3_val_labels,
    n_classes=2,
    embedding_dim=EMBEDDING_DIM,
    label_names=["2", "4"],
)


=== Final eval ===
Macro-F1: 0.7285


##N4

In [ ]:
n4_train = load_from_disk(os.path.join(DATA_PATH, "node/n4_train"))
n4_val = load_from_disk(os.path.join(DATA_PATH, "node/n4_val"))

n4_train_emb = precompute_embeddings(n4_train, encoder)
n4_val_emb = precompute_embeddings(n4_val, encoder)

n4_train_labels = [s["label"] for s in n4_train]
n4_val_labels = [s["label"] for s in n4_val]

print(f"[n4] Train: {len(n4_train_emb)}, distribution: {Counter(n4_train_labels)}")
print(f"[n4] Val:   {len(n4_val_emb)}, distribution: {Counter(n4_val_labels)}")

Encoding:   0%|          | 0/29 [00:00<?, ?it/s]

Encoding:   0%|          | 0/6 [00:00<?, ?it/s]

[n4] Train: 154, distribution: Counter({0: 84, 1: 70})
[n4] Val:   30, distribution: Counter({1: 15, 0: 15})


In [ ]:
n4_state, n4_best_f1, n4_history = train_node(
    train_embeddings=n4_train_emb,
    train_labels=n4_train_labels,
    val_embeddings=n4_val_emb,
    val_labels=n4_val_labels,
    n_classes=2,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=128,
    lr=1e-3,
    batch_size=16,
    max_epochs=100,
    patience=10,
    use_class_weights=True,
    seed=0,
)

[Train] Class weights: [0.9169999957084656, 1.100000023841858]
Epoch   1 | train_loss=0.7034 | val_macroF1=0.6997 ★
Epoch   2 | train_loss=0.6545 | val_macroF1=0.7321 ★
Epoch   3 | train_loss=0.6148 | val_macroF1=0.4750
Epoch   4 | train_loss=0.5864 | val_macroF1=0.4444
Epoch   5 | train_loss=0.5413 | val_macroF1=0.4444
Epoch   6 | train_loss=0.5078 | val_macroF1=0.5238
Epoch   7 | train_loss=0.4717 | val_macroF1=0.4994
Epoch   8 | train_loss=0.4344 | val_macroF1=0.4994
Epoch   9 | train_loss=0.4066 | val_macroF1=0.4750
Epoch  10 | train_loss=0.3861 | val_macroF1=0.4750
Epoch  11 | train_loss=0.3625 | val_macroF1=0.4750
Epoch  12 | train_loss=0.3488 | val_macroF1=0.4750
[Train] Early stopping at epoch 12 (best val F1=0.7321)


In [ ]:
n4_f1, n4_preds, n4_true = evaluate_node_full(
    n4_state, n4_val_emb, n4_val_labels,
    n_classes=2,
    embedding_dim=EMBEDDING_DIM,
    label_names=["3", "5"],
)


=== Final eval ===
Macro-F1: 0.7321


#Pipeline

In [ ]:
# Helper to predict with each node
@torch.no_grad()
def predict_node(model_state, embeddings, n_classes, embedding_dim,
                 hidden_dim=128, batch_size=16):

    model = GatedAttentionMIL(embedding_dim, n_classes, hidden_dim).to(DEVICE)
    model.load_state_dict(model_state)
    model.eval()

    ds = DialogueBagDataset(embeddings, [0] * len(embeddings))  # label dummy
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False, collate_fn=collate_bags)

    all_logits = []
    for padded, mask, _ in loader:
        padded, mask = padded.to(DEVICE), mask.to(DEVICE)
        logits, _ = model(padded, mask)
        all_logits.append(logits.cpu())

    logits = torch.cat(all_logits)
    probs = F.softmax(logits, dim=-1).numpy()
    preds = probs.argmax(axis=-1)
    return preds, probs

#  N1
#  ├── 0
#  ├── 8
#  └── defensive → N2
#                  ├── 1
#                  ├── 6
#                  ├── 7
#                  ├── ImgDist → N3
#                  |             ├── 2
#                  |             └── 4
#                  └── RealAv → N4
#                                ├── 3
#                                └── 5

def pipeline(samples, encoder,
               n1_state, n2_state, n3_state, n4_state,
               embedding_dim,
               n1_hidden=128, n2_hidden=128, n3_hidden=128, n4_hidden=128):



    n_samples = len(samples)
    final_preds = np.full(n_samples, -1, dtype=int)

    embeddings = precompute_embeddings(samples, encoder)

    # N1
    n1_preds, _ = predict_node(n1_state, embeddings, n_classes=3,
                                embedding_dim=embedding_dim, hidden_dim=n1_hidden)

    # Routing N1
    final_preds[n1_preds == 0] = 0
    final_preds[n1_preds == 1] = 8

    idx_to_n2 = np.where(n1_preds == 2)[0]

    if len(idx_to_n2) == 0:
        return final_preds

    # N2
    embeddings_n2 = [embeddings[i] for i in idx_to_n2]
    n2_preds, _ = predict_node(n2_state, embeddings_n2, n_classes=5,
                                embedding_dim=embedding_dim, hidden_dim=n2_hidden)

    # Routing N2
    # n2_pred == 0 → label 1
    # n2_pred == 1 → ImgDist (a N3)
    # n2_pred == 2 → RealAv (a N4)
    # n2_pred == 3 → label 6
    # n2_pred == 4 → label 7

    for local_idx, global_idx in enumerate(idx_to_n2):
        n2_p = n2_preds[local_idx]
        if n2_p == 0:
            final_preds[global_idx] = 1
        elif n2_p == 3:
            final_preds[global_idx] = 6
        elif n2_p == 4:
            final_preds[global_idx] = 7

    # N3
    idx_to_n3_local = np.where(n2_preds == 1)[0]
    idx_to_n3_global = idx_to_n2[idx_to_n3_local]

    if len(idx_to_n3_global) > 0:
        embeddings_n3 = [embeddings[i] for i in idx_to_n3_global]
        n3_preds, _ = predict_node(n3_state, embeddings_n3, n_classes=2,
                                    embedding_dim=embedding_dim, hidden_dim=n3_hidden)
        # n3_pred == 0 → class 2, n3_pred == 1 → class 4
        for local_idx, global_idx in enumerate(idx_to_n3_global):
            final_preds[global_idx] = 2 if n3_preds[local_idx] == 0 else 4

    # N4
    idx_to_n4_local = np.where(n2_preds == 2)[0]
    idx_to_n4_global = idx_to_n2[idx_to_n4_local]

    if len(idx_to_n4_global) > 0:
        embeddings_n4 = [embeddings[i] for i in idx_to_n4_global]
        n4_preds, _ = predict_node(n4_state, embeddings_n4, n_classes=2,
                                    embedding_dim=embedding_dim, hidden_dim=n4_hidden)
        # n4_pred == 0 → class 3, n4_pred == 1 → class 5
        for local_idx, global_idx in enumerate(idx_to_n4_global):
            final_preds[global_idx] = 3 if n4_preds[local_idx] == 0 else 5
    return final_preds



def evaluate_pipeline(pipeline_fn, samples, true_labels):

    preds = pipeline_fn(samples)
    true = np.array(true_labels)

    macro_f1 = f1_score(true, preds, average="macro", labels=list(range(9)),
                        zero_division=0)

    f1_per_class = f1_score(true, preds, average=None, labels=list(range(9)),
                             zero_division=0)

    return {
        "macro_f1": macro_f1,
        "per_class_f1": f1_per_class
    }

Eval

In [ ]:
val_full = load_from_disk(os.path.join(DATA_PATH, "split/val"))

val_full_labels = [s["label"] for s in val_full]

results = evaluate_pipeline(
    pipeline_fn=lambda s: pipeline(
        s, encoder,
        n1_state=n1_state,
        n2_state=n2_state,
        n3_state=n3_state,
        n4_state=n4_state,
        embedding_dim=EMBEDDING_DIM,
        n1_hidden=128, n2_hidden=128, n3_hidden=128, n4_hidden=128,
    ),
    samples=val_full,
    true_labels=val_full_labels,
)

print(f"\nMacro-F1: {results['macro_f1']:.4f}")

print(f"\nPer-class F1:")
for cls in range(9):
    f1_p1 = results['per_class_f1'][cls]
    print(f"  {cls} | {f1_p1:>8.4f}")

Encoding:   0%|          | 0/24 [00:00<?, ?it/s]


Macro-F1: 0.2802

Per-class F1:
  0 |   0.5455
  1 |   0.3571
  2 |   0.4118
  3 |   0.2222
  4 |   0.2143
  5 |   0.1667
  6 |   0.2963
  7 |   0.3077
  8 |   0.0000


Test

In [ ]:
test_full = load_from_disk(os.path.join(DATA_PATH, "split/test"))

test_full_labels = [s["label"] for s in test_full]

results = evaluate_pipeline(
    pipeline_fn=lambda s: pipeline(
        s, encoder,
        n1_state=n1_state,
        n2_state=n2_state,
        n3_state=n3_state,
        n4_state=n4_state,
        embedding_dim=EMBEDDING_DIM,
        n1_hidden=128, n2_hidden=128, n3_hidden=128, n4_hidden=128,
    ),
    samples=test_full,
    true_labels=test_full_labels,
)

print(f"\nMacro-F1: {results['macro_f1']:.4f}")

print(f"\nPer-class F1:")
for cls in range(9):
    f1_p1 = results['per_class_f1'][cls]
    print(f"  {cls} | {f1_p1:>8.4f}")

Encoding:   0%|          | 0/81 [00:00<?, ?it/s]


Macro-F1: 0.2580

Per-class F1:
  0 |   0.6977
  1 |   0.1111
  2 |   0.2449
  3 |   0.1224
  4 |   0.1587
  5 |   0.0845
  6 |   0.2439
  7 |   0.6591
  8 |   0.0000
